In [0]:
# Databricks Notebook: 01_Bronze_Ingestion


# Read all CSV files from raw container
raw_path = "abfss://raw@sarveshbankinglake00189.dfs.core.windows.net/"


accounts_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}accounts.csv")
atm_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}atm_transactions.csv")
branches_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}branches.csv")
credit_cards_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}credit_cards.csv")
customers_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}customers.csv")
employees_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}employees.csv")
fraud_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}fraud_alerts.csv")
kyc_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}kyc_documents.csv")
loans_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}loans.csv")
transactions_df = spark.read.option("header", True).option("inferSchema", True).csv(f"{raw_path}transactions.csv")


print("✅All CSV files loaded successfully")


In [0]:
%python
# Write each DataFrame as Delta table in Bronze layer
bronze_path = "abfss://bronze@sarveshbankinglake00189.dfs.core.windows.net/"


# Accounts
accounts_df.write.format("delta").mode("overwrite").save(f"{bronze_path}accounts")


# ATM Transactions
atm_df.write.format("delta").mode("overwrite").save(f"{bronze_path}atm_transactions")


# Branches
branches_df.write.format("delta").mode("overwrite").save(f"{bronze_path}branches")


# Credit Cards
credit_cards_df.write.format("delta").mode("overwrite").save(f"{bronze_path}credit_cards")


# Customers
customers_df.write.format("delta").mode("overwrite").save(f"{bronze_path}customers")


# Employees
employees_df.write.format("delta").mode("overwrite").save(f"{bronze_path}employees")


# Fraud Alerts
fraud_df.write.format("delta").mode("overwrite").save(f"{bronze_path}fraud_alerts")


# KYC Documents
kyc_df.write.format("delta").mode("overwrite").save(f"{bronze_path}kyc_documents")


# Loans
loans_df.write.format("delta").mode("overwrite").save(f"{bronze_path}loans")


# Transactions
transactions_df.write.format("delta").mode("overwrite").save(f"{bronze_path}transactions")


print("✅ All tables written to Bronze layer!")


In [0]:
%sql
-- Create external tables pointing to Bronze Delta files
CREATE TABLE banking_catalog.bronze.accounts
USING DELTA
LOCATION 'abfss://bronze@sarveshbankinglake00189.dfs.core.windows.net/accounts/';




In [0]:
%python
# Databricks Notebook: 02_Silver_Transformation


from pyspark.sql.functions import *
from pyspark.sql.types import *


# Read from Bronze
accounts_bronze = spark.table("banking_catalog.bronze.accounts")


# Data Cleaning Operations
accounts_silver = accounts_bronze \
    .withColumn("amount", coalesce(col("amount"), lit(0.0))) \
    .withColumn("status", when(col("status").isin(['ACTIVE', 'INACTIVE', 'PENDING']), col("status")).otherwise('UNKNOWN')) \
    .withColumn("type", when(col("type").isin(['LOAN', 'CURRENT', 'SAVINGS']), col("type")).otherwise('OTHER')) \
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
    .withColumn("remarks", coalesce(col("remarks"), lit("NO_REMARK"))) \
    .withColumn("flag", coalesce(col("flag"), lit("N"))) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("data_source", lit("accounts.csv"))


# Remove duplicates
accounts_silver = accounts_silver.dropDuplicates(["account_id"])


# Write to Silver
accounts_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/accounts")


print(f"✅ Accounts Silver: {accounts_silver.count()} records")



In [0]:

%python
# Read from Bronze
customers_bronze = spark.table("banking_catalog.bronze.customers")


# Data Cleaning
customers_silver = customers_bronze \
    .withColumn("name", trim(col("name"))) \
    .withColumn("gender", when(col("gender").isin(['M', 'F']), col("gender")).otherwise('U')) \
    .withColumn("phone", regexp_replace(col("phone"), "[^0-9]", "")) \
    .withColumn("email", lower(trim(col("email")))) \
    .withColumn("city", initcap(trim(col("city")))) \
    .withColumn("state", upper(trim(col("state")))) \
    .withColumn("kyc_status", when(col("kyc_status").isin(['Y', 'N']), col("kyc_status")).otherwise('N')) \
    .withColumn("dob", to_date(col("dob"), "yyyy-MM-dd")) \
    .withColumn("created_date", to_date(col("created_date"), "yyyy-MM-dd")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("data_source", lit("customers.csv"))


# Handle NULLs
customers_silver = customers_silver.fillna({
    "phone": "0000000000",
    "email": "unknown@email.com",
    "kyc_status": "N"
})


# Remove duplicates
customers_silver = customers_silver.dropDuplicates(["customer_id"])


# Write to Silver
customers_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/customers")


print(f"✅ Customers Silver: {customers_silver.count()} records")

In [0]:
# Problem 8: Clean and Transform Transactions Data

# Read from Bronze
transactions_bronze = spark.table("banking_catalog.bronze.transactions")

# Data Cleaning
transactions_silver = transactions_bronze \
    .withColumn("amount", coalesce(col("amount"), lit(0.0))) \
    .withColumn("status", when(col("status").isin(['ACTIVE', 'INACTIVE', 'PENDING']), col("status")).otherwise('UNKNOWN')) \
    .withColumn("type", when(col("type").isin(['LOAN', 'CURRENT', 'SAVINGS']), col("type")).otherwise('OTHER')) \
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
    .withColumn("remarks", coalesce(col("remarks"), lit("NO_REMARK"))) \
    .withColumn("flag", coalesce(col("flag"), lit("N"))) \
    .withColumn("year", year(col("date"))) \
    .withColumn("month", month(col("date"))) \
    .withColumn("quarter", quarter(col("date"))) \
    .withColumn("ingestion_timestamp", current_timestamp())

# Filter invalid transactions (negative amount or amount > 100000)
transactions_silver = transactions_silver \
    .filter(col("amount") >= 0) \
    .filter(col("amount") <= 100000) \
    .dropDuplicates(["transaction_id"])

# Write to Silver
transactions_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/transactions")

print(f"✅ Transactions Silver: {transactions_silver.count()} records")

In [0]:
# Problem 9: Clean and Transform Loans Data

# Read from Bronze
loans_bronze = spark.table("banking_catalog.bronze.loans")

# Data Cleaning
loans_silver = loans_bronze \
    .withColumn("amount", coalesce(col("amount"), lit(0.0))) \
    .withColumn("status", when(col("status").isin(['ACTIVE', 'INACTIVE', 'PENDING']), col("status")).otherwise('UNKNOWN')) \
    .withColumn("type", when(col("type").isin(['LOAN', 'SAVINGS', 'CURRENT']), col("type")).otherwise('OTHER')) \
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
    .withColumn("remarks", coalesce(col("remarks"), lit("NO_REMARK"))) \
    .withColumn("flag", coalesce(col("flag"), lit("N"))) \
    .withColumn("ingestion_timestamp", current_timestamp())

# Remove duplicates
loans_silver = loans_silver.dropDuplicates(["loan_id"])

# Write to Silver
loans_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/loans")

print(f"✅ Loans Silver: {loans_silver.count()} records")

In [0]:
# Problem 10: Clean and Transform Fraud Alerts Data

# Read from Bronze
fraud_bronze = spark.table("banking_catalog.bronze.fraud_alerts")

# Data Cleaning
fraud_silver = fraud_bronze \
    .withColumn("amount", coalesce(col("amount"), lit(0.0))) \
    .withColumn("status", when(col("status").isin(['ACTIVE', 'INACTIVE', 'PENDING']), col("status")).otherwise('UNKNOWN')) \
    .withColumn("type", when(col("type").isin(['LOAN', 'CURRENT', 'SAVINGS']), col("type")).otherwise('OTHER')) \
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
    .withColumn("remarks", coalesce(col("remarks"), lit("NO_REMARK"))) \
    .withColumn("flag", coalesce(col("flag"), lit("N"))) \
    .withColumn("is_high_risk", when(col("amount") > 40000, lit("Y")).otherwise(lit("N"))) \
    .withColumn("ingestion_timestamp", current_timestamp())

# Remove duplicates
fraud_silver = fraud_silver.dropDuplicates(["alert_id"])

# Write to Silver
fraud_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/fraud_alerts")

print(f"✅ Fraud Alerts Silver: {fraud_silver.count()} records")

In [0]:
%sql
-- Problem 11: Create Silver Tables in Unity Catalog
--Create Silver tables

CREATE TABLE banking_catalog.silver.accounts
USING DELTA
LOCATION 'abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/accounts/';

CREATE TABLE banking_catalog.silver.customers
USING DELTA
LOCATION 'abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/customers/';

CREATE TABLE banking_catalog.silver.transactions
USING DELTA
LOCATION 'abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/transactions/';

CREATE TABLE banking_catalog.silver.loans
USING DELTA
LOCATION 'abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/loans/';

CREATE TABLE banking_catalog.silver.fraud_alerts
USING DELTA
LOCATION 'abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/fraud_alerts/';

CREATE TABLE banking_catalog.silver.atm_transactions
USING DELTA
LOCATION 'abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/atm_transactions/';

CREATE TABLE banking_catalog.silver.branches
USING DELTA
LOCATION 'abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/branches/';

CREATE TABLE banking_catalog.silver.credit_cards
USING DELTA
LOCATION 'abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/credit_cards/';

CREATE TABLE banking_catalog.silver.kyc_documents
USING DELTA
LOCATION 'abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/kyc_documents/';

CREATE TABLE banking_catalog.silver.employees
USING DELTA
LOCATION 'abfss://silver@sarveshbankinglake00189.dfs.core.windows.net/employees/';

In [0]:
#PART 3: GOLD LAYER - Business Insights & Analytics
#Problem 12: Create Dim_Customers Table (SCD Type 2)%

# Databricks Notebook: 03_Gold_Star_Schema

from pyspark.sql.window import Window

# Read Silver tables
customers_silver = spark.table("banking_catalog.silver.customers")
accounts_silver = spark.table("banking_catalog.silver.accounts")
transactions_silver = spark.table("banking_catalog.silver.transactions")
loans_silver = spark.table("banking_catalog.silver.loans")
fraud_silver = spark.table("banking_catalog.silver.fraud_alerts")
atm_silver = spark.table("banking_catalog.silver.atm_transactions")
branches_silver = spark.table("banking_catalog.silver.branches")

In [0]:
from pyspark.sql.functions import *
# Create Dim_Customers with SCD Type 2
dim_customers = customers_silver \
    .withColumn("customer_sk", monotonically_increasing_id()) \
    .withColumn("effective_date", current_date()) \
    .withColumn("end_date", lit("9999-12-31").cast("date")) \
    .withColumn("is_current", lit(True)) \
    .withColumn("full_name", concat(col("name"), lit(" (ID: "), col("customer_id"), lit(")"))) \
    .withColumn("age", datediff(current_date(), col("dob")) / 365.25) \
    .withColumn("age_group", 
        when(col("age") < 30, "Young")
        .when(col("age") < 50, "Middle")
        .otherwise("Senior"))

# Write to Gold
dim_customers.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/dim_customers")

print(f"✅ Dim_Customers: {dim_customers.count()} records")

In [0]:
#Problem 13: Create Dim_Accounts Table

# Create Dim_Accounts
dim_accounts = accounts_silver \
    .withColumn("account_sk", monotonically_increasing_id()) \
    .withColumn("effective_date", current_date()) \
    .withColumn("is_current", lit(True)) \
    .withColumn("account_age_days", datediff(current_date(), col("date"))) \
    .withColumn("balance_category",
        when(col("amount") < 10000, "Low")
        .when(col("amount") < 50000, "Medium")
        .otherwise("High"))

# Write to Gold
dim_accounts.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/dim_accounts")

print(f"✅ Dim_Accounts: {dim_accounts.count()} records")

In [0]:
#Problem 14: Create Dim_Date Table
# Create Date Dimension (2020-2025)
def create_date_dimension(start_date, end_date):
    dates = spark.sql(f"""
        SELECT EXPLODE(SEQUENCE(
            TO_DATE('{start_date}'), 
            TO_DATE('{end_date}'), 
            INTERVAL 1 DAY
        )) AS date
    """)
    
    date_dim = dates.select(
        col("date"),
        year("date").alias("year"),
        quarter("date").alias("quarter"),
        month("date").alias("month"),
        dayofmonth("date").alias("day"),
        dayofweek("date").alias("day_of_week"),
        date_format("date", "EEEE").alias("day_name"),
        date_format("date", "MMMM").alias("month_name"),
        when(dayofweek("date").isin([1, 7]), "Weekend").otherwise("Weekday").alias("weekday_indicator")
    ).withColumn("date_sk", monotonically_increasing_id())
    
    return date_dim

date_dim_df = create_date_dimension("2020-01-01", "2025-12-31")

date_dim_df.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/dim_date")

print(f"✅ Dim_Date: {date_dim_df.count()} records")

In [0]:
#Problem 15: Create Fact_Transactions

# Create Fact_Transactions with surrogate keys
fact_transactions = transactions_silver \
    .join(dim_accounts.select("account_id", "account_sk"), "account_id", "left") \
    .join(dim_customers.select("customer_id", "customer_sk"), "customer_id", "left") \
    .join(date_dim_df.select("date", "date_sk"), transactions_silver["date"] == date_dim_df["date"], "left") \
    .select(
        "transaction_id",
        "customer_sk",
        "account_sk",
        "date_sk",
        "amount",
        "status",
        col("type").alias("transaction_type"),
        "branch",
        "remarks",
        "flag",
        "year",
        "month",
        "quarter"
    )

# Write to Gold
fact_transactions.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/fact_transactions")

print(f"✅ Fact_Transactions: {fact_transactions.count()} records")

In [0]:
#Problem 16: Create Fact_Loans

# Create Fact_Loans
fact_loans = loans_silver \
    .join(dim_accounts.select("account_id", "account_sk"), "account_id", "left") \
    .join(dim_customers.select("customer_id", "customer_sk"), "customer_id", "left") \
    .join(date_dim_df.select("date", "date_sk"), loans_silver["date"] == date_dim_df["date"], "left") \
    .select(
        "loan_id",
        "customer_sk",
        "account_sk",
        "date_sk",
        "amount",
        "status",
        col("type").alias("loan_type"),
        "branch",
        "remarks",
        "flag"
    )

# Write to Gold
fact_loans.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/fact_loans")

print(f"✅ Fact_Loans: {fact_loans.count()} records")

In [0]:
#Problem 17: Create Fact_Fraud_Alerts

# Create Fact_Fraud_Alerts
fact_fraud = fraud_silver \
    .join(dim_customers.select("customer_id", "customer_sk"), "customer_id", "left") \
    .join(dim_accounts.select("account_id", "account_sk"), "account_id", "left") \
    .join(date_dim_df.select("date", "date_sk"), fraud_silver["date"] == date_dim_df["date"], "left") \
    .select(
        "alert_id",
        "customer_sk",
        "account_sk",
        "date_sk",
        "amount",
        "status",
        col("type").alias("alert_type"),
        "branch",
        "remarks",
        "flag",
        "is_high_risk"
    )

# Write to Gold
fact_fraud.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true")\
    .save("abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/fact_fraud_alerts")

print(f"✅ Fact_Fraud_Alerts: {fact_fraud.count()} records")

In [0]:
%sql

-- Create Gold Tables in Unity Catalog
-- Create Gold tables
CREATE TABLE banking_catalog.gold.dim_customers
USING DELTA
LOCATION 'abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/dim_customers/';

CREATE TABLE banking_catalog.gold.dim_accounts
USING DELTA
LOCATION 'abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/dim_accounts/';

CREATE TABLE banking_catalog.gold.dim_date
USING DELTA
LOCATION 'abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/dim_date/';

CREATE TABLE banking_catalog.gold.fact_transactions
USING DELTA
LOCATION 'abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/fact_transactions/';

CREATE TABLE banking_catalog.gold.fact_loans
USING DELTA
LOCATION 'abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/fact_loans/';

CREATE TABLE banking_catalog.gold.fact_fraud_alerts
USING DELTA
LOCATION 'abfss://gold@sarveshbankinglake00189.dfs.core.windows.net/fact_fraud_alerts/';

In [0]:
%sql
--PART 4: 30+ BUSINESS PROBLEMS SOLVED
--Problem 20: Total Balance by Account Type
-- Problem: What is the total balance for each account type?
SELECT 
    type,
    COUNT(*) as account_count,
    SUM(amount) as total_balance,
    AVG(amount) as avg_balance,
    MIN(amount) as min_balance,
    MAX(amount) as max_balance
FROM banking_catalog.silver.accounts
WHERE amount IS NOT NULL
GROUP BY type
ORDER BY total_balance DESC;

In [0]:
%sql
--Problem 21: Customer KYC Status Analysis
-- Problem: What percentage of customers have completed KYC?
SELECT 
    kyc_status,
    COUNT(*) as customer_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM banking_catalog.silver.customers
GROUP BY kyc_status
ORDER BY percentage DESC;

In [0]:
%sql
--Problem 22: Monthly Transaction Trends
-- Problem: How do transaction volumes trend monthly?
SELECT 
    YEAR(date) as year,
    MONTH(date) as month,
    COUNT(*) as transaction_count,
    SUM(amount) as total_amount,
    AVG(amount) as avg_amount
FROM banking_catalog.silver.transactions
GROUP BY YEAR(date), MONTH(date)
ORDER BY year DESC, month DESC;

In [0]:
%sql
--Problem 23: Top 10 Customers by Transaction Volume
-- Problem: Who are the top 10 customers by transaction volume?
SELECT 
    c.customer_id,
    c.name,
    COUNT(t.transaction_id) as transaction_count,
    SUM(t.amount) as total_transaction_amount,
    AVG(t.amount) as avg_transaction
FROM banking_catalog.silver.customers c
JOIN banking_catalog.silver.transactions t
ON c.customer_id = t.customer_id
GROUP BY c.customer_id, c.name
ORDER BY total_transaction_amount DESC
LIMIT 10;

In [0]:
%sql
-- Problem 24: Branch Performance Comparison
-- Problem: How does each branch perform in terms of transactions?
SELECT 
    branch,
    COUNT(*) as transaction_count,
    SUM(amount) as total_amount,
    AVG(amount) as avg_amount,
    COUNT(DISTINCT customer_id) as unique_customers
FROM banking_catalog.silver.transactions
GROUP BY branch
ORDER BY total_amount DESC;

In [0]:
%sql
-- Problem 25: Fraud Alert Status Distribution
-- Problem: What is the distribution of fraud alert statuses?
SELECT 
    status,
    COUNT(*) as alert_count,
    SUM(amount) as total_amount,
    AVG(amount) as avg_amount
FROM banking_catalog.silver.fraud_alerts
GROUP BY status
ORDER BY alert_count DESC;

In [0]:
%sql
-- Problem 26: High-Risk Fraud Alerts
-- Problem: Identify high-risk fraud alerts (>40000)
SELECT 
    alert_id,
    customer_id,
    amount,
    status,
    type,
    date,
    branch
FROM banking_catalog.silver.fraud_alerts
WHERE amount > 40000
ORDER BY amount DESC;

In [0]:
%sql
--Problem 27: Loan Portfolio Status
-- Problem: What is the status of the loan portfolio?
SELECT 
    status,
    COUNT(*) as loan_count,
    SUM(amount) as total_loan_amount,
    AVG(amount) as avg_loan_amount,
    ROUND(SUM(amount) * 100.0 / SUM(SUM(amount)) OVER(), 2) as percentage
FROM banking_catalog.silver.loans
GROUP BY status
ORDER BY total_loan_amount DESC;

In [0]:
%sql
-- Problem 28: Customer Gender Distribution
-- Problem: What is the gender distribution of customers?
SELECT 
    COALESCE(gender, 'Unknown') as gender,
    COUNT(*) as customer_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM banking_catalog.silver.customers
GROUP BY gender
ORDER BY percentage DESC;

In [0]:
%sql
--Problem 29: Average Transaction Amount by City
-- Problem: Which cities have the highest average transaction amounts?
SELECT 
    c.city,
    COUNT(t.transaction_id) as transaction_count,
    ROUND(AVG(t.amount), 2) as avg_transaction_amount,
    ROUND(SUM(t.amount), 2) as total_transaction_amount
FROM banking_catalog.silver.customers c
JOIN banking_catalog.silver.transactions t
ON c.customer_id = t.customer_id
WHERE c.city IS NOT NULL
GROUP BY c.city
ORDER BY avg_transaction_amount DESC
LIMIT 10;

In [0]:
%sql
-- Problem 30: Active vs Inactive Accounts
-- Problem: What is the ratio of active vs inactive accounts?
SELECT 
    status,
    COUNT(*) as account_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM banking_catalog.silver.accounts
GROUP BY status
ORDER BY percentage DESC;

In [0]:
%sql
-- Problem 32: Customer Lifecycle - Created Date Analysis
-- Problem: When were most customers created?
SELECT 
    YEAR(created_date) as year,
    MONTH(created_date) as month,
    COUNT(*) as new_customers
FROM banking_catalog.silver.customers
WHERE created_date IS NOT NULL
GROUP BY YEAR(created_date), MONTH(created_date)
ORDER BY year DESC, month DESC
LIMIT 12;

In [0]:
%sql
-- Problem 34: Customers with Both Loans and Accounts
-- Problem: Which customers have both loans and active accounts?
SELECT DISTINCT
    c.customer_id,
    c.name,
    c.city
FROM banking_catalog.silver.customers c
WHERE EXISTS (
    SELECT 1 FROM banking_catalog.silver.accounts a 
    WHERE a.customer_id = c.customer_id AND a.status = 'ACTIVE'
)
AND EXISTS (
    SELECT 1 FROM banking_catalog.silver.loans l 
    WHERE l.customer_id = c.customer_id
);

In [0]:
%sql

-- Problem 35: Fraudulent Transactions vs Regular Transactions
-- Problem: How do fraudulent transactions compare to regular ones?
SELECT 
    'Regular' as transaction_type,
    COUNT(*) as count,
    ROUND(AVG(amount), 2) as avg_amount,
    ROUND(SUM(amount), 2) as total_amount
FROM banking_catalog.silver.transactions
UNION ALL
SELECT 
    'Fraud' as transaction_type,
    COUNT(*) as count,
    ROUND(AVG(amount), 2) as avg_amount,
    ROUND(SUM(amount), 2) as total_amount
FROM banking_catalog.silver.fraud_alerts;

In [0]:
%sql
-- Problem 36: Quarterly Performance Analysis
-- Problem: How does the bank perform each quarter?
SELECT 
    YEAR(date) as year,
    QUARTER(date) as quarter,
    COUNT(*) as transaction_count,
    SUM(amount) as total_transactions,
    AVG(amount) as avg_transaction
FROM banking_catalog.silver.transactions
GROUP BY YEAR(date), QUARTER(date)
ORDER BY year DESC, quarter DESC;

In [0]:
%sql

-- Problem 37: Top 5 Branches by Customer Base
-- Problem: Which branches have the most customers?
SELECT 
    branch,
    COUNT(DISTINCT customer_id) as unique_customers,
    COUNT(*) as transaction_count
FROM banking_catalog.silver.transactions
GROUP BY branch
ORDER BY unique_customers DESC
LIMIT 5;

In [0]:
%sql
-- Problem 38: KYC Verification Status by City
-- Problem: How does KYC status vary across cities?
SELECT 
    city,
    COUNT(*) as total_customers,
    SUM(CASE WHEN kyc_status = 'Y' THEN 1 ELSE 0 END) as kyc_completed,
    ROUND(SUM(CASE WHEN kyc_status = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as kyc_rate
FROM banking_catalog.silver.customers
WHERE city IS NOT NULL
GROUP BY city
ORDER BY kyc_rate DESC;

In [0]:
%sql
-- Problem 39: Loan Default Risk Analysis
-- Problem: Identify customers with high loan default risk
SELECT 
    c.customer_id,
    c.name,
    COUNT(l.loan_id) as total_loans,
    SUM(l.amount) as total_loan_amount,
    ROUND(AVG(l.amount), 2) as avg_loan_amount,
    MAX(l.amount) as max_loan_amount
FROM banking_catalog.silver.customers c
JOIN banking_catalog.silver.loans l
ON c.customer_id = l.customer_id
WHERE l.status = 'PENDING'
GROUP BY c.customer_id, c.name
HAVING SUM(l.amount) > 100000
ORDER BY total_loan_amount DESC;

In [0]:
%sql
--Problem 40: Daily Transaction Pattern
-- Problem: What is the transaction pattern by day of week?
SELECT 
    DAYOFWEEK(date) as day_number,
    CASE DAYOFWEEK(date)
        WHEN 1 THEN 'Sunday'
        WHEN 2 THEN 'Monday'
        WHEN 3 THEN 'Tuesday'
        WHEN 4 THEN 'Wednesday'
        WHEN 5 THEN 'Thursday'
        WHEN 6 THEN 'Friday'
        WHEN 7 THEN 'Saturday'
    END as day_name,
    COUNT(*) as transaction_count,
    ROUND(AVG(amount), 2) as avg_amount
FROM banking_catalog.silver.transactions
GROUP BY DAYOFWEEK(date)
ORDER BY day_number;

In [0]:
%sql
-- Problem 42: Cross-Sell Analysis - Customers with Multiple Products
-- Problem: Which customers have the most banking products?
SELECT 
    c.customer_id,
    c.name,
    COUNT(DISTINCT a.account_id) as total_accounts,
    COUNT(DISTINCT l.loan_id) as total_loans,
    COUNT(DISTINCT cc.card_id) as total_credit_cards,
    COUNT(DISTINCT f.alert_id) as fraud_alerts
FROM banking_catalog.silver.customers c
LEFT JOIN banking_catalog.silver.accounts a ON c.customer_id = a.customer_id
LEFT JOIN banking_catalog.silver.loans l ON c.customer_id = l.customer_id
LEFT JOIN banking_catalog.silver.credit_cards k ON c.customer_id = k.customer_id
LEFT JOIN banking_catalog.silver.fraud_alerts f ON c.customer_id = f.customer_id
GROUP BY c.customer_id, c.name
ORDER BY total_accounts DESC, total_loans DESC
LIMIT 10;

In [0]:
%sql
--Problem 43: Branch Loan Performance error
-- Problem: How do branches perform on loan approvals?
SELECT 
    branch,
    COUNT(*) as total_loans,
    SUM(CASE WHEN status = 'ACTIVE' THEN 1 ELSE 0 END) as active_loans,
    SUM(amount) as total_loan_amount,
    ROUND(AVG(amount), 2) as avg_loan_amount,
    ROUND(SUM(CASE WHEN status = 'ACTIVE' THEN amount ELSE 0 END) * 100.0 / SUM(amount), 2) as active_loan_rate
FROM banking_catalog.silver.loans
GROUP BY branch
ORDER BY total_loan_amount DESC;

In [0]:
%sql
--Problem 44: Customer Wealth Segmentation
-- Problem: How is customer wealth distributed?
SELECT 
    CASE 
        WHEN total_balance < 10000 THEN 'Low (< 10K)'
        WHEN total_balance < 50000 THEN 'Medium (10K - 50K)'
        WHEN total_balance < 100000 THEN 'High (50K - 100K)'
        ELSE 'Very High (> 100K)'
    END as wealth_segment,
    COUNT(*) as customer_count,
    ROUND(AVG(total_balance), 2) as avg_balance,
    ROUND(SUM(total_balance), 2) as total_wealth
FROM (
    SELECT 
        customer_id,
        SUM(amount) as total_balance
    FROM banking_catalog.silver.accounts
    GROUP BY customer_id
) customer_balances
GROUP BY wealth_segment
ORDER BY wealth_segment;

In [0]:
%sql
-- Problem 45: Recent Fraud Trends
-- Problem: How is fraud trending over recent months?
SELECT 
    DATE_TRUNC('month', date) as month,
    COUNT(*) as fraud_count,
    ROUND(AVG(amount), 2) as avg_fraud_amount,
    SUM(amount) as total_fraud_amount
FROM banking_catalog.silver.fraud_alerts
WHERE date >= DATE_SUB(CURRENT_DATE(), 180)
GROUP BY DATE_TRUNC('month', date)
ORDER BY month DESC;

In [0]:
%sql
--Problem 46: Customer Acquisition by State
-- Problem: Which states have the highest customer acquisition?
SELECT 
    state,
    COUNT(*) as total_customers,
    SUM(CASE WHEN kyc_status = 'Y' THEN 1 ELSE 0 END) as kyc_verified,
    DATE_TRUNC('year', created_date) as acquisition_year
FROM banking_catalog.silver.customers
WHERE state IS NOT NULL
GROUP BY state, DATE_TRUNC('year', created_date)
ORDER BY acquisition_year DESC, total_customers DESC;

In [0]:
%sql
-- Problem 47: Transaction Amount Distribution
-- Problem: How are transaction amounts distributed?
SELECT 
    CASE 
        WHEN amount < 1000 THEN 'Micro (< 1K)'
        WHEN amount < 10000 THEN 'Small (1K - 10K)'
        WHEN amount < 50000 THEN 'Medium (10K - 50K)'
        ELSE 'Large (> 50K)'
    END as transaction_segment,
    COUNT(*) as transaction_count,
    ROUND(AVG(amount), 0) as avg_amount,
    ROUND(SUM(amount), 0) as total_amount,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM banking_catalog.silver.transactions
GROUP BY transaction_segment
ORDER BY transaction_segment;

In [0]:
%sql
--Problem 48: Customer Activity Patterns
-- Problem: What are the activity patterns of customers?
WITH customer_stats AS (
    SELECT 
        customer_id,
        COUNT(*) as total_transactions,
        SUM(amount) as total_spent,
        DATEDIFF(CURRENT_DATE(), MAX(date)) as days_since_last_transaction
    FROM banking_catalog.silver.transactions
    GROUP BY customer_id
)
SELECT 
    CASE 
        WHEN total_transactions >= 100 THEN 'Very Active'
        WHEN total_transactions >= 50 THEN 'Active'
        WHEN total_transactions >= 10 THEN 'Moderate'
        ELSE 'Occasional'
    END as activity_level,
    COUNT(*) as customer_count,
    ROUND(AVG(total_spent), 2) as avg_spent,
    ROUND(AVG(days_since_last_transaction), 2) as avg_days_inactive
FROM customer_stats
GROUP BY activity_level
ORDER BY activity_level;

In [0]:
%sql
--Problem 50: YOY -Year-over-Year Analysis - Customer Growth
-- Problem: How has customer base grown year over year?
SELECT 
    YEAR(created_date) as year,
    COUNT(*) as new_customers,
    LAG(COUNT(*)) OVER (ORDER BY YEAR(created_date)) as prev_year_customers,
    ROUND((COUNT(*) - LAG(COUNT(*)) OVER (ORDER BY YEAR(created_date))) * 100.0 / LAG(COUNT(*)) OVER (ORDER BY YEAR(created_date)), 2) as yoy_growth
FROM banking_catalog.silver.customers
WHERE created_date IS NOT NULL
GROUP BY YEAR(created_date)
ORDER BY year;

In [0]:
%sql
--PART 5: OPTIMIZATION & MAINTENANCE
--Problem 51: Optimize Delta Tables


-- Optimize all tables for better performance
OPTIMIZE banking_catalog.silver.accounts;
OPTIMIZE banking_catalog.silver.transactions;
OPTIMIZE banking_catalog.silver.loans;
OPTIMIZE banking_catalog.gold.fact_transactions;
OPTIMIZE banking_catalog.gold.fact_loans;

In [0]:
%sql
--Problem 52: Vacuum Old Files
-- Clean up old files (retention period 7 days)
VACUUM banking_catalog.silver.transactions RETAIN 168 HOURS;
VACUUM banking_catalog.gold.fact_transactions RETAIN 168 HOURS;
VACUUM banking_catalog.bronze.transactions RETAIN 168 HOURS;

In [0]:
%sql
--Problem 53: Collect Table Statistics
-- Collect statistics for query optimization
ANALYZE TABLE banking_catalog.silver.accounts COMPUTE STATISTICS;
ANALYZE TABLE banking_catalog.silver.transactions COMPUTE STATISTICS;
ANALYZE TABLE banking_catalog.gold.fact_transactions COMPUTE STATISTICS;